In [35]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [36]:
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash") 

In [37]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(..., description="Detailed feedback for the essay.")
    score: int = Field(..., description="Score for the essay on a scale of 1 to 10.", ge=0, le=10)

In [38]:
structured_model = model.with_structured_output(EvaluationSchema)

In [39]:
essay= """
**Machine Learning**

Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve their performance without being explicitly programmed for every task. It focuses on developing algorithms that can identify patterns, analyze information, and make predictions or decisions based on previous experiences. Machine learning is widely used in various fields, including healthcare, education, finance, transportation, and cybersecurity. For example, in healthcare, machine learning models can help doctors detect diseases by analyzing medical images and patient records. In finance, it can identify suspicious transactions and detect potential fraud. There are three major types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled datasets to train models, while unsupervised learning discovers hidden patterns in unlabeled data. Reinforcement learning allows an agent to learn through interactions with an environment and feedback in the form of rewards or penalties. With the advancement of computing power and the increasing availability of data, machine learning has become an essential technology in modern society. Although it offers numerous opportunities, challenges such as data privacy, algorithmic bias, and model accuracy remain important concerns. Overall, machine learning is transforming the way humans solve complex problems and shaping the future of technology."""

In [40]:
prompt = f"""Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 (in essay).

{essay}"""

structured_model.invoke(prompt)

EvaluationSchema(feedback='The language quality of this essay is excellent. It demonstrates exceptional clarity, conciseness, and precision in its explanations. The vocabulary used is appropriate and sophisticated without being overly technical, making the complex topic of Machine Learning accessible to a broad audience. Sentence structure is varied, contributing to a smooth and engaging flow. There are no noticeable grammatical errors or spelling mistakes, indicating a high level of linguistic proficiency. The essay is well-organized, with logical transitions between ideas, from definition and applications to types, challenges, and its overall impact. It effectively conveys a comprehensive overview of Machine Learning in a professional and articulate manner.', score=9)

In [41]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]

In [42]:
def evaluate_language(state: UPSCState):
    prompt = f"""Evaluate the language quality of the following essay. Assess grammar, vocabulary, sentence structure, and readability. Provide detailed feedback and assign a score out of 10.

{state['essay']}"""

    output = structured_model.invoke(prompt)

    return {
        "language_feedback": output.feedback,
        "individual_scores": [output.score]
    }

def evaluate_analysis(state: UPSCState):
    prompt = f"""Evaluate the depth of analysis in the following essay. Assess the strength of arguments, evidence, reasoning, and critical insight. Provide detailed feedback and assign a score out of 10.

{state['essay']}"""

    output = structured_model.invoke(prompt)

    return {
        "analysis_feedback": output.feedback,
        "individual_scores": [output.score]
    }

def evaluate_thought(state: UPSCState):
    prompt = f"""Evaluate the clarity of thought in the following essay. Assess organization, coherence, logical flow, and how clearly the central ideas are presented. Provide detailed feedback and assign a score out of 10.

{state['essay']}"""

    output = structured_model.invoke(prompt)

    return {
        "clarity_feedback": output.feedback,
        "individual_scores": [output.score]
    }

def final_evaluation(state: UPSCState):
    prompt = f"""Based on the following feedback, provide an overall evaluation of the essay.

Language feedback: {state['language_feedback']}
Analysis feedback: {state['analysis_feedback']}
Clarity feedback: {state['clarity_feedback']}
Individual scores: {state['individual_scores']}

Summarize the essay's strengths and weaknesses and assign a final score out of 10."""

    output = structured_model.invoke(prompt)

    return {
        "overall_feedback": output.feedback,
        "individual_scores": [output.score]
    }

In [45]:
graph = StateGraph(UPSCState)

graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_thought", evaluate_thought)
graph.add_node("final_evaluation", final_evaluation)

# Run the three essay evaluations in parallel
graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_thought")

# Final evaluation runs after all three evaluations complete
graph.add_edge([
    "evaluate_language",
    "evaluate_analysis",
    "evaluate_thought",
], "final_evaluation")

graph.add_edge("final_evaluation", END)

workflow = graph.compile()


In [46]:
initial_state = {
    "essay": essay,
}

workflow.invoke(initial_state)

{'essay': '\n**Machine Learning**\n\nMachine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data and improve their performance without being explicitly programmed for every task. It focuses on developing algorithms that can identify patterns, analyze information, and make predictions or decisions based on previous experiences. Machine learning is widely used in various fields, including healthcare, education, finance, transportation, and cybersecurity. For example, in healthcare, machine learning models can help doctors detect diseases by analyzing medical images and patient records. In finance, it can identify suspicious transactions and detect potential fraud. There are three major types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled datasets to train models, while unsupervised learning discovers hidden patterns in unlabeled data. Reinforcement learning allow